# 📊 Anomaly Detection on MVTec-AD Dataset Evaluation Dashboard
This notebook allows you to evaluate anomaly detection results of an off-the-shelf method against the MVTec-AD dataset.

**You do not need to connect a GPU runtime, CPU env is fine but it will take around 3,5 hours on a CPU env.**

This notebook calculates AP, AUPR, F1, and AUROC on Image (_I) and Pixel (_P) levels, and also PRO metric on Pixel-level.  


It mainly relies on `metrics` package in `sklearn`, with some preprocessing of the predictions by using average pooling to smooth them.

**1. 🛠️ Setup**

We will install required packages for the evaluation metrics and fetch ground truth data from Official website.

In [ ]:
!pip -q install numpy pandas scikit-learn scikit-image tabulate pillow opencv-python gdown

**2. Evaluation Helper Code**

Main metric evaluation code.

In [ ]:
import glob
import logging
import os

import numpy as np
import copy
import tabulate
import torch
import torch.nn.functional as F
from sklearn import metrics
from sklearn.metrics import auc, average_precision_score, precision_recall_curve
from skimage import measure
import pandas as pd
from numpy import ndarray
from statistics import mean
np.seterr(divide='ignore',invalid='ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class EvalDataMeta:
    def __init__(self, preds, masks, file_info=None):
        self.preds = preds  # N x H x W
        self.masks = masks  # N x H x W
        self.file_info = file_info


class EvalImage:
    def __init__(self, data_meta, **kwargs):
        self.preds = self.encode_pred(data_meta.preds, **kwargs)
        self.masks = self.encode_mask(data_meta.masks)
        self.file_info = data_meta.file_info
        self.preds_good = sorted(self.preds[self.masks == 0], reverse=True)
        self.preds_defe = sorted(self.preds[self.masks == 1], reverse=True)
        self.num_good = len(self.preds_good)
        self.num_defe = len(self.preds_defe)
        self.desc_score_indices = np.argsort(self.preds, kind="mergesort")[::-1]
        self.y_score = self.preds[self.desc_score_indices]
        self.y_true = self.masks == 1
        self.y_true = self.y_true[self.desc_score_indices]
        self.y_true2 = self.y_true[self.desc_score_indices]

    @staticmethod
    def encode_pred(preds):
        raise NotImplementedError

    def encode_mask(self, masks):
        N, _, _ = masks.shape
        masks = (masks.reshape(N, -1).sum(axis=1) != 0).astype(np.int_)  # (N, )
        return masks

    def eval_auc(self):
        fpr, tpr, thresholds = metrics.roc_curve(self.masks, self.preds, pos_label=1)
        auc = metrics.auc(fpr, tpr)
        if auc < 0.5:
            auc = 1 - auc
        return auc


class EvalImageMean(EvalImage):
    @staticmethod
    def encode_pred(preds):
        N, _, _ = preds.shape
        return preds.reshape(N, -1).mean(axis=1)  # (N, )


class EvalImageStd(EvalImage):
    @staticmethod
    def encode_pred(preds):
        N, _, _ = preds.shape
        return preds.reshape(N, -1).std(axis=1)  # (N, )


def top_k_pooling(preds, k_ratio=0.01):
    """
    Select top-k% pixels and average them for image-level score.
    """
    if preds.ndim == 4:
        preds = preds[:, 0, :, :]
    flat_preds = preds.reshape(preds.shape[0], -1)
    k = max(1, int(flat_preds.shape[1] * k_ratio))
    topk_scores = np.partition(flat_preds, -k, axis=1)[:, -k:]
    return topk_scores.mean(axis=1)

class EvalImageMax(EvalImage):
    @staticmethod
    def encode_pred(preds):
        N, _, _ = preds.shape
        preds = torch.tensor(preds[:, None, ...])  # N x 1 x H x W
        preds.to(device)
        for i in range(0, 8):
            preds = (F.avg_pool2d(preds, 8, stride=1))
        preds = preds.cpu().numpy()  # N x 1 x H x W
        return preds.reshape(N, -1).max(axis=1)  # (N, )

class EvalPerPixelAUC:
    def __init__(self, data_meta):
        self.preds = np.concatenate(
            [pred.flatten() for pred in data_meta.preds], axis=0
        )
        self.masks = np.concatenate(
            [mask.flatten() for mask in data_meta.masks], axis=0
        )
        self.masks[self.masks > 0] = 1

    def eval_auc(self):
        fpr, tpr, thresholds = metrics.roc_curve(self.masks, self.preds, pos_label=1)
        auc = metrics.auc(fpr, tpr)
        if auc < 0.5:
            auc = 1 - auc
        return auc

class EvalPerPixelPRO:
    def __init__(self, data_meta):
        self.preds = data_meta.preds
        self.masks = data_meta.masks
        self.masks[self.masks > 0] = 1

    def eval_auc(self):
        pro = compute_pro(self.masks, self.preds)
        return pro

class EvalPerPixelAP:
    def __init__(self, data_meta):
        self.preds = np.concatenate(
            [pred.flatten() for pred in data_meta.preds], axis=0
        )
        self.masks = np.concatenate(
            [mask.flatten() for mask in data_meta.masks], axis=0
        )
        self.masks[self.masks > 0] = 1
    def eval_auc(self):
        ap = average_precision_score(self.masks, self.preds)
        return ap
class EvalImageAP(EvalImage):
    @staticmethod
    def encode_pred(preds):
        N, _, _ = preds.shape
        preds = torch.tensor(preds[:, None, ...])#.cuda() # N x 1 x H x W
        preds = preds.to(device)
        for i in range(0, 8):
            preds = (F.avg_pool2d(preds, 8, stride=1))
        preds = preds.cpu().numpy()  # N x 1 x H x W
        return preds.reshape(N, -1).max(axis=1)  # (N, )
    def eval_auc(self):
        ap = average_precision_score(self.masks, self.preds)
        return ap
class EvalPerPixelF1:
    def __init__(self, data_meta):
        self.preds = np.concatenate(
            [pred.flatten() for pred in data_meta.preds], axis=0
        )
        self.masks = np.concatenate(
            [mask.flatten() for mask in data_meta.masks], axis=0
        )
        self.masks[self.masks > 0] = 1
    def eval_auc(self):
        precisions, recalls, thresholds = precision_recall_curve(self.masks, self.preds)
        f1_scores = (2 * precisions * recalls) / (precisions + recalls)
        f1_px = np.max(f1_scores[np.isfinite(f1_scores)])
        return f1_px
class EvalImageF1(EvalImage):
    @staticmethod
    def encode_pred(preds):
        N, _, _ = preds.shape
        preds = torch.tensor(preds[:, None, ...])  # N x 1 x H x W
        preds = preds.to(device)
        for i in range(0, 8):
            preds = (F.avg_pool2d(preds, 8, stride=1))
        preds = preds.cpu().numpy()  # N x 1 x H x W
        return preds.reshape(N, -1).max(axis=1)  # (N, )
    def eval_auc(self):
        precisions, recalls, thresholds = precision_recall_curve(self.masks, self.preds)
        f1_scores = (2 * precisions * recalls) / (precisions + recalls)
        f1_sp = np.max(f1_scores[np.isfinite(f1_scores)])
        return f1_sp
class EvalPerPixelAUPR:
    def __init__(self, data_meta):
        self.preds = np.concatenate(
            [pred.flatten() for pred in data_meta.preds], axis=0
        )
        self.masks = np.concatenate(
            [mask.flatten() for mask in data_meta.masks], axis=0
        )
        self.masks[self.masks > 0] = 1
    def eval_auc(self):
        pr_auc = compute_aupr(self.preds, self.masks)
        return pr_auc
class EvalImageAUPR(EvalImage):
    @staticmethod
    def encode_pred(preds):
        N, _, _ = preds.shape
        preds = torch.tensor(preds[:, None, ...])  # N x 1 x H x W
        preds = preds.to(device)
        for i in range(0, 8):
            preds = (F.avg_pool2d(preds, 8, stride=1))
        preds = (F.avg_pool2d(preds, 2, stride=1))
        preds = preds.cpu().numpy()  # N x 1 x H x W
        return preds.reshape(N, -1).max(axis=1)  # (N, )
    def eval_auc(self):
        pr_auc = compute_aupr(self.preds, self.masks)
        return pr_auc

eval_lookup_table = {
    "mean": EvalImageMean,
    "std": EvalImageStd,
    "max": EvalImageMax,
    "pixel": EvalPerPixelAUC,
    "pro" :EvalPerPixelPRO,
    "appx": EvalPerPixelAP,
    "apsp": EvalImageAP,
    "f1px": EvalPerPixelF1,
    "f1sp": EvalImageF1,
    "auprpx": EvalPerPixelAUPR,
    "auprsp": EvalImageAUPR,
}

def performances(fileinfos, preds, masks, config):
    ret_metrics = {}
    clsnames = set([fileinfo["clsname"] for fileinfo in fileinfos])
    for clsname in clsnames:
        preds_cls = []
        masks_cls = []
        file_cls = []
        for fileinfo, pred, mask in zip(fileinfos, preds, masks):
            if fileinfo["clsname"] == clsname:
                preds_cls.append(pred[None, ...])
                masks_cls.append(mask[None, ...])
                file_cls.append(fileinfo['filename'])
        preds_cls = np.concatenate(np.asarray(preds_cls), axis=0)  # N x H x W
        masks_cls = np.concatenate(np.asarray(masks_cls), axis=0)  # N x H x W
        data_meta = EvalDataMeta(preds_cls, masks_cls, file_cls)

        # auc
        if config.get("auc", None):
            for metric in config["auc"]:
                evalname = metric["name"]
                kwargs = metric.get("kwargs", {})
                eval_method = eval_lookup_table[evalname](data_meta, **kwargs)
                auc = eval_method.eval_auc()
                ret_metrics["{}_{}_auc".format(clsname, evalname)] = auc

    if config.get("auc", None):
        for metric in config["auc"]:
            evalname = metric["name"]
            evalvalues = [
                ret_metrics["{}_{}_auc".format(clsname, evalname)]
                for clsname in clsnames
            ]
            mean_auc = np.mean(np.array(evalvalues))
            ret_metrics["{}_{}_auc".format("mean", evalname)] = mean_auc

    return ret_metrics

def compute_pro(masks: ndarray, amaps: ndarray, num_th: int = 200) -> None:

    """Compute the area under the curve of per-region overlaping (PRO) and 0 to 0.3 FPR
    Args:
        category (str): Category of product
        masks (ndarray): All binary masks in test. masks.shape -> (num_test_data, h, w)
        amaps (ndarray): All anomaly maps in test. amaps.shape -> (num_test_data, h, w)
        num_th (int, optional): Number of thresholds
    """
    assert isinstance(amaps, ndarray), "type(amaps) must be ndarray"
    assert isinstance(masks, ndarray), "type(masks) must be ndarray"
    assert amaps.ndim == 3, "amaps.ndim must be 3 (num_test_data, h, w)"
    assert masks.ndim == 3, "masks.ndim must be 3 (num_test_data, h, w)"
    assert amaps.shape == masks.shape, "amaps.shape and masks.shape must be same"
    assert set(masks.flatten()) == {0, 1}, "set(masks.flatten()) must be {0, 1}"
    assert isinstance(num_th, int), "type(num_th) must be int"
    df = pd.DataFrame([], columns=["pro", "fpr", "threshold"])
    binary_amaps = np.zeros_like(amaps, dtype=np.bool_)
    min_th = amaps.min()
    max_th = amaps.max()
    delta = (max_th - min_th) / num_th
    for th in np.arange(min_th, max_th, delta):
        binary_amaps[amaps <= th] = 0
        binary_amaps[amaps > th] = 1
        pros = []
        for binary_amap, mask in zip(binary_amaps, masks):
            for region in measure.regionprops(measure.label(mask)):
                axes0_ids = region.coords[:, 0]
                axes1_ids = region.coords[:, 1]
                tp_pixels = binary_amap[axes0_ids, axes1_ids].sum()
                pros.append(tp_pixels / region.area)
        inverse_masks = 1 - masks
        fp_pixels = np.logical_and(inverse_masks, binary_amaps).sum()
        fpr = fp_pixels / inverse_masks.sum()
        df_new = pd.DataFrame([{"pro": mean(pros), "fpr": fpr, "threshold": th}])
        df = pd.concat([df, df_new], ignore_index=True)
        # df = df.append({"pro": mean(pros), "fpr": fpr, "threshold": th}, ignore_index=True)
    # Normalize FPR from 0 ~ 1 to 0 ~ 0.3
    df = df[df["fpr"] < 0.3]
    df["fpr"] = df["fpr"] / df["fpr"].max()
    pro_auc = auc(df["fpr"], df["pro"])
    return pro_auc

def compute_aupr(
    predicted_masks,
    ground_truth_masks,
    include_optimal_threshold_rates=False,
):
    """
    Computes pixel-wise statistics (AUROC, FPR, TPR) for anomaly segmentations
    and ground truth segmentation masks.

    Args:
        predicted_masks: [list of np.arrays or np.array] [NxHxW] Contains
                               generated segmentation masks.
        ground_truth_masks: [list of np.arrays or np.array] [NxHxW] Contains
                            predefined ground truth segmentation masks
    """
    pred_mask = copy.deepcopy(predicted_masks)
    gt_mask = copy.deepcopy(ground_truth_masks)
    num = 200
    out = {}

    if pred_mask is None or gt_mask is None:
        for key in out:
            out[key].append(float('nan'))
    else:
        fprs, tprs = [], []
        precisions, f1s = [], []
        gt_mask = np.array(gt_mask, np.uint8)

        t = (gt_mask == 1)
        f = ~t
        n_true = t.sum()
        n_false = f.sum()
        th_min = pred_mask.min() - 1e-8
        th_max = pred_mask.max() + 1e-8
        pred_gt = pred_mask[t]
        th_gt_min = pred_gt.min()
        th_gt_max = pred_gt.max()

        '''
        Using scikit learn to compute pixel au_roc results in a memory error since it tries to store the NxHxW float score values.
        To avoid this, we compute the tp, fp, tn, fn at equally spaced thresholds in the range between min of predicted
        scores and maximum of predicted scores
        '''
        percents = np.linspace(100, 0, num=num // 2)
        th_gt_per = np.percentile(pred_gt, percents)
        th_unif = np.linspace(th_gt_max, th_gt_min, num=num // 2)
        thresholds = np.concatenate([th_gt_per, th_unif, [th_min, th_max]])
        thresholds = np.flip(np.sort(thresholds))

        if n_true == 0 or n_false == 0:
            raise ValueError("gt_submasks must contains at least one normal and anomaly samples")

        for th in thresholds:
            p = (pred_mask > th).astype(np.uint8)
            p = (p == 1)
            fp = (p & f).sum()
            tp = (p & t).sum()

            fpr = fp / n_false
            tpr = tp / n_true
            if tp + fp > 0:
                prec = tp / (tp + fp)
            else:
                prec = 1.0
            if prec > 0. and tpr > 0.:
                f1 = (2 * prec * tpr) / (prec + tpr)
            else:
                f1 = 0.0
            fprs.append(fpr)
            tprs.append(tpr)
            precisions.append(prec)

        pr_auc = metrics.auc(tprs, precisions)
        pr_auc = round(pr_auc, 4)

    return pr_auc

3. Upload the dataset from your Drive or local directory.

The official website of [MVTec-AD](https://www.mvtec.com/research-teaching/datasets/mvtec-ad/downloads) gives a download [link](https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420938113-1629960298/mvtec_anomaly_detection.tar.xz).

In [ ]:
# Option 2: Use a download URL
# replace with your official signed/direct URL if you have one
MVTEC_URL = "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420938113-1629960298/mvtec_anomaly_detection.tar.xz"
!mkdir -p /content/data
!wget -O /content/mvtec_anomaly_detection.tar.xz "$MVTEC_URL"
!tar -xJf /content/mvtec_anomaly_detection.tar.xz -C /content/data


4. Upload predictions.zip file. Example prediction file is here: https://storage.googleapis.com/predictions_colab/predictions.zip


In [ ]:
# Option 1: Upload from local
# from google.colab import files
# uploaded = files.upload()

!rm -rf /content/predictions_npz

import zipfile
from pathlib import Path
import os

!wget -O /content/predictions.zip https://storage.googleapis.com/predictions_colab/predictions.zip

ZIP_PATH = Path("/content/predictions.zip")   # TODO: change if your uploaded file has a different name
PRED_DIR = Path("/content/predictions_npz/")

assert ZIP_PATH.exists(), f"Zip file not found: {ZIP_PATH}"

PRED_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(PRED_DIR)

print("Extracted to:", PRED_DIR)

PRED_DIR = Path("/content/predictions_npz/predictions")

!rm -rf /content/predictions_npz/__MACOSX

npz_files = sorted(PRED_DIR.rglob("*.npz"))

print("num npz files:", len(npz_files))
print("first 10:")
for p in npz_files[:10]:
    print(" ", p)

5. Dataset Preprocessing

In [ ]:
import numpy as np
from PIL import Image
from pathlib import Path

MVTEC_ROOT = Path("/content/data")

MVTEC_CLASSES = sorted([
    p.name for p in MVTEC_ROOT.iterdir()
    if p.is_dir()
])

print("classes:", MVTEC_CLASSES)

def load_official_mask(mask_path, target_size=None):
    mask = np.array(Image.open(mask_path).convert("L"))
    mask = (mask > 0).astype(np.uint8)

    if target_size is not None and (mask.shape[1], mask.shape[0]) != target_size:
        import cv2
        mask = cv2.resize(mask, target_size, interpolation=cv2.INTER_NEAREST)
        mask = (mask > 0).astype(np.uint8)

    return mask


def build_official_mvtec_index_light(mvtec_root: Path):
    index = {}

    for cls_dir in sorted([p for p in mvtec_root.iterdir() if p.is_dir()]):
        clsname = cls_dir.name
        test_root = cls_dir / "test"
        gt_root = cls_dir / "ground_truth"

        for defect_dir in sorted([p for p in test_root.iterdir() if p.is_dir()]):
            defect_type = defect_dir.name

            for img_path in sorted(defect_dir.glob("*")):
                if img_path.suffix.lower() not in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}:
                    continue

                rel_test_path = str(img_path.relative_to(cls_dir)).replace("\\", "/")

                if defect_type == "good":
                    gt_path = None
                else:
                    gt_path = gt_root / defect_type / f"{img_path.stem}_mask.png"
                    if not gt_path.exists():
                        raise FileNotFoundError(f"Missing GT mask: {gt_path}")

                index[(clsname, rel_test_path)] = {
                    "image_path": str(img_path.resolve()),
                    "gt_path": None if gt_path is None else str(gt_path.resolve()),
                    "clsname": clsname,
                    "rel_test_path": rel_test_path,
                }

    return index

official_index = build_official_mvtec_index_light(MVTEC_ROOT)
print("official samples:", len(official_index))

first_items = list(official_index.items())[:5]
for k, v in first_items:
    print(k, v["image_path"])

6. Load predictions data to dict

In [ ]:
import os
import cv2
import numpy as np
from collections import defaultdict

def to_python_str(x):
    if isinstance(x, bytes):
        return x.decode("utf-8")
    if isinstance(x, np.ndarray) and x.shape == ():
        x = x.item()
        if isinstance(x, bytes):
            return x.decode("utf-8")
        return str(x)
    return str(x)

def load_official_mask(gt_path, image_path):
    if gt_path is None:
        img = Image.open(image_path).convert("RGB")
        w, h = img.size
        return np.zeros((h, w), dtype=np.uint8)

    mask = np.array(Image.open(gt_path).convert("L"))
    return (mask > 0).astype(np.uint8)

def normalize_single_pred(pred):
    pred = np.asarray(pred)
    if pred.ndim == 3:
        if pred.shape[0] == 1:
            pred = pred[0]
        elif pred.shape[-1] == 1:
            pred = pred[..., 0]
        else:
            raise ValueError(f"Unsupported pred shape: {pred.shape}")
    elif pred.ndim != 2:
        raise ValueError(f"Prediction must be 2D or singleton-channel 3D, got {pred.shape}")
    return pred.astype(np.float32)

def candidate_rel_paths(filename_str, clsname):
    """
    Generate possible relative paths like:
      test/broken_large/000.png
    from filename strings that may be absolute or partial.
    """
    s = str(filename_str).replace("\\", "/")
    candidates = set()

    # exact as-is
    candidates.add(s)

    # strip everything before "<clsname>/"
    token = f"{clsname}/"
    if token in s:
        candidates.add(s.split(token, 1)[1])

    # strip everything before "test/"
    if "/test/" in s:
        candidates.add(s.split("/test/", 1)[1])
        candidates.add("test/" + s.split("/test/", 1)[1])

    # basename only won't be enough globally, but keep for debugging
    candidates.add(os.path.basename(s))

    return candidates


pred_files_by_class = defaultdict(list)

for npz_path in npz_files:
    data = np.load(npz_path, allow_pickle=True)
    clsname = to_python_str(data["clsname"])
    pred_files_by_class[clsname].append(npz_path)

print(sorted((k, len(v)) for k, v in pred_files_by_class.items()))

7. Calculate the Metrics

In [ ]:
import cv2
import gc
import numpy as np
from PIL import Image

config = {
    "auc": [
        {"name": "pixel"},
        {"name": "pro"},
        {"name": "appx"},
        {"name": "f1px"},
        {"name": "auprpx"},
        {"name": "max"},
        {"name": "apsp"},
        {"name": "f1sp"},
        {"name": "auprsp"},
    ]
}

all_metrics = {}

for clsname in sorted(pred_files_by_class.keys()):
    cls_records = []

    for npz_path in pred_files_by_class[clsname]:
        data = np.load(npz_path, allow_pickle=True)

        pred = normalize_single_pred(data["pred"])
        filename = to_python_str(data["filename"])

        matched = None
        for rel_candidate in candidate_rel_paths(filename, clsname):
            key = (clsname, rel_candidate)
            if key in official_index:
                matched = official_index[key]
                break

        if matched is None:
            print(f"Skipping unmatched file: {npz_path.name}")
            continue

        gt = load_official_mask(matched["gt_path"], matched["image_path"])

        if pred.shape != gt.shape:
            pred = cv2.resize(pred, (gt.shape[1], gt.shape[0]), interpolation=cv2.INTER_LINEAR)

        cls_records.append({
            "filename": matched["image_path"],
            "clsname": clsname,
            "rel_test_path": matched["rel_test_path"],
            "pred": pred,
            "mask": gt,
        })

    cls_records = sorted(cls_records, key=lambda x: x["rel_test_path"])

    fileinfos = [{"filename": r["filename"], "clsname": r["clsname"]} for r in cls_records]
    preds = np.stack([r["pred"] for r in cls_records], axis=0).astype(np.float32)
    masks = np.stack([r["mask"] for r in cls_records], axis=0).astype(np.uint8)

    cls_metrics = performances(fileinfos, preds, masks, config)
    all_metrics.update(cls_metrics)

    print(f"done: {clsname}")

    del cls_records, fileinfos, preds, masks, cls_metrics
    gc.collect()

all_metrics

8. Print the results

In [ ]:
import pandas as pd

rows = []
for k, v in all_metrics.items():
    for clsname in MVTEC_CLASSES:
        prefix = clsname + "_"
        if k.startswith(prefix):
            metric_name = k[len(prefix):]
            rows.append({
                "class": clsname,
                "metric": metric_name,
                "value": v,
            })
            break

df = pd.DataFrame(rows)

table = df.pivot(index="class", columns="metric", values="value")

metric_order = ["pixel", "pro", "appx", "f1px", "auprpx", "max", "apsp", "f1sp", "auprsp"]
existing_cols = [c for c in metric_order if c in table.columns]
other_cols = [c for c in table.columns if c not in existing_cols]
table = table[existing_cols + other_cols]

existing_rows = [c for c in MVTEC_CLASSES if c in table.index]
other_rows = [c for c in table.index if c not in existing_rows]
table = table.reindex(existing_rows + other_rows)

# add mean row at the end
table.loc["mean"] = table.mean(axis=0)

rename_map = {
    "appx_auc": "AP_P",
    "apsp_auc": "AP_I",
    "auprpx_auc": "AUPR_P",
    "auprsp_auc": "AUPR_I",
    "f1px_auc": "F1_P",
    "f1sp_auc": "F1_I",
    "max_auc": "AUROC_I",
    "pixel_auc": "AUROC_P",
    "pro_auc": "PRO_P",
}

table_pretty = table.rename(columns=rename_map)

col_order = ["AP_P", "AP_I", "AUPR_P", "AUPR_I", "F1_P", "F1_I", "AUROC_I", "AUROC_P", "PRO_P"]
existing_cols = [c for c in col_order if c in table_pretty.columns]
other_cols = [c for c in table_pretty.columns if c not in existing_cols]
table_pretty = table_pretty[existing_cols + other_cols]

table_pretty.style.format("{:.4f}")